In [ ]:
import os
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Set plotting styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

In [ ]:
def get_predictions(EMBEDDING_MODEL_NAME):
    db_path = os.path.join("..", "data", "goodreads.db")
    print(f"Connecting to database: {db_path}")
    conn = sqlite3.connect(db_path)

    if EMBEDDING_MODEL_NAME:
        model_filter = f"WHERE em.name = '{EMBEDDING_MODEL_NAME}'"
    else:
        model_filter = "WHERE em.id = (SELECT embedding_model_id FROM book_predictions ORDER BY date_updated DESC LIMIT 1)"

    # Query to load predictions joined with book metadata and user's read/rating status.
    # Joins embedding_models so predictions are scoped to a single model.
    # We join library_books via best_book_lookup to ensure that if you read/rated any edition
    # of a book, it is mapped to the canonical bestBook record correctly.
    query = f"""
    SELECT 
        bp.book_id,
        em.name AS embedding_model,
        b.title,
        b.description,
        b.num_pages,
        b.language_name,
        b.web_url,
        b.publisher,
        bp.solo_pred_rating,
        bp.friend_pred_rating,
        bp.count_adjusted_rating,
        bp.final_rating,
        bp.date_updated,
        MAX(lb.rating) as my_rating,
        MAX(CASE WHEN lb.book_legacy_id IS NOT NULL THEN 1 ELSE 0 END) as in_library,
        MAX(CASE WHEN lb.rating IS NOT NULL AND lb.rating > 0 THEN 1 ELSE 0 END) as is_rated_by_me,
        (b.star_1 + b.star_2 + b.star_3 + b.star_4 + b.star_5) as ratings_count,
        (b.star_1 * 1.0 + b.star_2 * 2.0 + b.star_3 * 3.0 + b.star_4 * 4.0 + b.star_5 * 5.0) /
            NULLIF(b.star_1 + b.star_2 + b.star_3 + b.star_4 + b.star_5, 0) as avg_rating
    FROM book_predictions bp
    JOIN embedding_models em ON em.id = bp.embedding_model_id
    JOIN books b ON bp.book_id = b.legacy_id
    LEFT JOIN best_book_lookup bbl ON bp.book_id = bbl.best_book_id
    LEFT JOIN library_books lb ON bbl.raw_legacy_id = lb.book_legacy_id
        AND lb.library_id = (SELECT legacy_id FROM libraries WHERE is_main = 1)
    {model_filter}
    GROUP BY bp.book_id, bp.embedding_model_id
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    model_label = df['embedding_model'].iloc[0] if not df.empty else 'N/A'
    print(f"Loaded {len(df)} predictions for model: {model_label}.")

    return df[df["is_rated_by_me"] == 0].copy().set_index("book_id")

cols = ["title", "count_adjusted_rating", "solo_pred_rating", "friend_pred_rating", "final_rating", "num_pages"]

In [ ]:
# corr_cols = [
#     "solo_pred_rating",
#     "friend_pred_rating",
#     "count_adjusted_rating",
#     "final_rating",
#     "avg_rating",
#     "my_rating",
# ]
# corr_df = df[corr_cols].corr()

# plt.figure(figsize=(10, 8))
# sns.heatmap(corr_df, annot=True, cmap="coolwarm", fmt=".3f", vmin=-1, vmax=1)
# plt.title("Correlation Matrix of Predictions and Ratings")
# plt.show()

In [ ]:
# # friend_pred_rating is NULL when no friend libraries were seeded — drop those rows.
# friend_df = df.dropna(subset=["friend_pred_rating"]).copy()
# if friend_df.empty:
#     print("No friend predictions available (friend_pred_rating is NULL for all rows). Run friend_similarity first.")
# else:
#     plt.figure(figsize=(10, 8))
#     sns.scatterplot(
#         data=friend_df,
#         x="solo_pred_rating",
#         y="friend_pred_rating",
#         hue="is_rated_by_me",
#         palette={0: "gray", 1: "blue"},
#         alpha=0.6,
#     )
#     plt.title("Solo vs. Friend Predicted Ratings")
#     plt.xlabel("Solo Predicted Rating")
#     plt.ylabel("Friend Predicted Rating")
#     plt.legend(title="Rated by Me")
#     plt.show()

## 50

In [ ]:
df = get_predictions("leoipulsar/harrier-0.6b:latest")
df[cols].sort_values(by="final_rating", ascending=False)

In [ ]:
df = get_predictions("qwen3-embedding:0.6b")
df[cols].sort_values(by="final_rating", ascending=False)

## 100

In [ ]:
df = get_predictions("leoipulsar/harrier-0.6b:latest")
df[cols].sort_values(by="final_rating", ascending=False)

In [ ]:
df = get_predictions("qwen3-embedding:0.6b")
df[cols].sort_values(by="final_rating", ascending=False)

## 200

In [ ]:
df = get_predictions("leoipulsar/harrier-0.6b:latest")
df[cols].sort_values(by="final_rating", ascending=False)

In [ ]:
df = get_predictions("qwen3-embedding:0.6b")
df[cols].sort_values(by="final_rating", ascending=False)

## 300

In [ ]:
df = get_predictions("leoipulsar/harrier-0.6b:latest")
df[cols].sort_values(by="final_rating", ascending=False)

In [ ]:
df = get_predictions("qwen3-embedding:0.6b")
df[cols].sort_values(by="final_rating", ascending=False)